# UK Biobank locus examples

In [2]:
file = list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong/", full.names = T)

In [3]:
res = readRDS(file[22])

In [4]:
file[22]

[1] "/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong//chr22:17336757-18536672.Monocyte_perc.rds"

In [10]:
   library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

# ---- fit loop, one output per (gene, tissue) ----
files = list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong", full.names = T, recursive = T)


f = files[22]

d <- readRDS(f)

insample_z  <- d[["insample_z"]]
insample_LD <- d[["insample_LD"]]
z_sketch    <- d[["z_sketch"]]
sketch      <- d[["sketch"]]
sketch_LD   <- d[["sketch_LD"]]
n           <- d[["n"]]
B_insample  <- d[["B_insample"]]
B_sketch    <- d[["B_sketch"]]

# ---- in-sample panel ----
insample_res         <- safe_susie(susie_hc, "insample_res",
                                   z = insample_z, R = insample_LD, n = n)
susie_plot(insample_res, "PIP")

insample_res$sets$cs

susie_rss_res        <- safe_susie(susie_hc, "susie_rss_res",
                                       z = z_sketch, X = sketch, n = n)

susie_eb_res         <- safe_susie(susie_hc, "susie_eb_res",
                                   z = z_sketch, X = sketch, n = n,
                                   R_finite = B_sketch, R_mismatch = "eb_mix")


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.




$L1
[1] 1051

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: R uncertainty inflation modifies per-variant SER likelihoods, which prevents a consistent model-level ELBO.



plot without title

In [15]:
insample_res$sets$cs

$L1
[1] 1051

In [11]:
susie_rss_res$sets$cs

$L1
[1] 1051

$L3
[1] 1032

$L2
[1]  960  997 1010

In [12]:
susie_eb_res$sets$cs

$L1
[1] 1051

In [13]:
susie_eb_res$R_finite_diagnostics$R_reliability_flag

[1] FALSE

In [14]:
susie_eb_res$R_finite_diagnostics$prior_variance_flag

[1] FALSE

In [18]:
library(tidyverse)

## ---- pull the real numbers from d + your fits ----
z_all   <- d[["z_sketch"]]      # z used by the mismatched (out-of-sample) fit
LD_ref  <- d[["sketch_LD"]]     # out-of-sample LD
LD_true <- d[["insample_LD"]]   # in-sample LD

lead <- which.max(abs(z_all))                       # anchor / lead variant
## spurious CS = credible set(s) in the out-of-sample fit that do NOT contain the lead
sp_cs  <- susie_rss_res$sets$cs[!vapply(susie_rss_res$sets$cs,
                                        function(m) lead %in% m, logical(1))]
sp_idx <- unlist(sp_cs, use.names = FALSE)
tag    <- sp_idx[which.max(abs(z_all[sp_idx]))]     # driver of the spurious CS

z_lead <- z_all[lead];  z_tag <- z_all[tag]
r_ref  <- LD_ref[lead,  tag]                        # out-of-sample LD(lead, tag)
r_true <- LD_true[lead, tag]                        # in-sample LD(lead, tag)

cond_z <- function(z_tag, z_lead, r, lambda = 0) (z_tag - r * z_lead) / sqrt(1 - r^2 + lambda)

cat(sprintf("lead=%d  z_lead=%.2f | tag=%d  z_tag=%.2f | r_in=%.3f  r_out=%.3f\n",
            lead, z_lead, tag, z_tag, r_true, r_ref))
cat(sprintf("residual | out-of-sample = %.2f (spurious) ;  in-sample = %.2f\n",
            cond_z(z_tag, z_lead, r_ref), cond_z(z_tag, z_lead, r_true)))

## optional real EB inflation — find the field, then set it (leave NA to skip the EB bar):
# str(susie_eb_res$R_finite_diagnostics)
lambda_eb <- NA_real_   # e.g. susie_eb_res$R_finite_diagnostics$lambda

## ---- build figure data from the real values ----
df <- tibble(
  stage = c("Marginal",
            sprintf("Cond. on lead\n(out-of-sample, r = %.2f)", r_ref),
            sprintf("Cond. on lead\n(in-sample, r = %.2f)", r_true)),
  z    = c(z_tag, cond_z(z_tag, z_lead, r_ref), cond_z(z_tag, z_lead, r_true)),
  kind = c("Marginal", "Out-of-sample", "In-sample")
)
if (!is.na(lambda_eb)) {
  df <- add_row(df,
    stage = "Cond. on lead + EB\n(variance inflated)",
    z     = cond_z(z_tag, z_lead, r_ref, lambda_eb),
    kind  = "EB-corrected", .after = 2)
}
df <- df %>% mutate(stage = factor(stage, levels = stage))

kind_cols <- c("Marginal" = "#7F7F7F", "Out-of-sample" = "#E6820E",
               "EB-corrected" = "#08306B", "In-sample" = "#238B45")

fw <- 9; fh <- 5
options(repr.plot.width = fw, repr.plot.height = fh)

p_leak <- ggplot(df, aes(stage, z, fill = kind)) +
  geom_col(width = 0.68) +
  geom_hline(yintercept = 2, linetype = "dashed", colour = "red", linewidth = 0.8) +
  annotate("text", x = Inf, y = 2.2, label = "\u2248 credible-set threshold",
           colour = "red", hjust = 1.02, size = 4.6) +
  geom_text(aes(label = sprintf("%.1f", z)), vjust = -0.4, size = 4.6, colour = "grey20") +
  scale_fill_manual(values = kind_cols, guide = "none") +
  labs(x = NULL, y = "Secondary-variant |z|") +
  coord_cartesian(ylim = c(0, max(df$z) * 1.15)) +
  theme_classic(base_size = 16) +
  theme(
    axis.title = element_text(size = 18),
    axis.text  = element_text(size = 13),
    axis.line  = element_line(linewidth = 0.6, colour = "black"),
    axis.ticks = element_line(linewidth = 0.6, colour = "black")
  )
p_leak

# ggsave("fig5d_leak_real.png", p_leak, width = fw, height = fh, dpi = 300)

lead=1051  z_lead=30.83 | tag=1032  z_tag=11.72 | r_in=0.313  r_out=0.220
residual | out-of-sample = 5.06 (spurious) ;  in-sample = 2.17


plot without title

In [1]:
library(tidyverse)
library(patchwork)

lead <- which.max(abs(insample_z))   # top-z variant (the anchor, ~index 1051)

tidy_pip <- function(fit, label, lead) {
  pip <- as.numeric(fit$pip); cs <- fit$sets$cs
  type <- rep(NA_character_, length(pip))
  if (!is.null(cs)) for (nm in names(cs)) {
    members <- cs[[nm]]
    type[members] <- if (lead %in% members) "Lead CS" else "Spurious CS"
  }
  tibble(variant = seq_along(pip), pip = pip, cs_type = type,
         panel = label, n_cs = if (is.null(cs)) 0L else length(cs))
}

pip_df <- bind_rows(
  tidy_pip(insample_res,  "In-sample LD",     lead),
  tidy_pip(susie_rss_res, "Out-of-sample LD", lead),
  tidy_pip(susie_eb_res,  "EB-corrected",     lead)
) %>%
  mutate(panel = factor(panel, levels = c("In-sample LD", "Out-of-sample LD", "EB-corrected")))

lab_map <- pip_df %>% distinct(panel, n_cs) %>%
  mutate(lab = sprintf("%s  (%d CS)", panel, n_cs)) %>% { setNames(.$lab, .$panel) }

cs_cols <- c("Lead CS" = "#08306B", "Spurious CS" = "#E6820E")

fw <- 9; fh <- 8
options(repr.plot.width = fw, repr.plot.height = fh)

p_pip <- ggplot(pip_df, aes(variant, pip)) +
  geom_point(data = filter(pip_df, is.na(cs_type)), colour = "grey78", size = 1) +
  geom_point(data = filter(pip_df, !is.na(cs_type)), aes(colour = cs_type), size = 2.6) +
  facet_grid(panel ~ ., labeller = labeller(panel = lab_map)) +
  scale_colour_manual(values = cs_cols, name = NULL) +
  labs(x = "Variant index", y = "PIP") +
  coord_cartesian(ylim = c(0, 1)) +
  theme_bw(base_size = 16) +
  theme(
    legend.position  = "bottom",
    axis.title       = element_text(size = 18),
    axis.text        = element_text(size = 14),
    strip.text       = element_text(size = 14, face = "bold"),
    strip.background = element_rect(fill = "grey95", colour = NA),
    panel.grid.minor = element_blank()
  )
p_pip

# ggsave("fig5d_pip_tracks.png", p_pip, width = fw, height = fh, dpi = 300)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


plot without title

## Additional locus example

In [4]:
file = list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong/", full.names = T)
res = readRDS(file[12])

In [5]:
max(abs(res$insample_z))

[1] 9.34842

In [6]:
file[12]

[1] "/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong//chr12:95606906-96464566.HLR_perc.rds"

In [7]:
   library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

# ---- fit loop, one output per (gene, tissue) ----
files = list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong", full.names = T, recursive = T)


f = files[12]

d <- readRDS(f)

insample_z  <- d[["insample_z"]]
insample_LD <- d[["insample_LD"]]
z_sketch    <- d[["z_sketch"]]
sketch      <- d[["sketch"]]
sketch_LD   <- d[["sketch_LD"]]
n           <- d[["n"]]
B_insample  <- d[["B_insample"]]
B_sketch    <- d[["B_sketch"]]

# ---- in-sample panel ----
insample_res         <- safe_susie(susie_hc, "insample_res",
                                   z = insample_z, R = insample_LD, n = n)
susie_plot(insample_res, "PIP")

insample_res$sets$cs

susie_rss_res        <- safe_susie(susie_hc, "susie_rss_res",
                                       z = z_sketch, X = sketch, n = n)

susie_eb_res         <- safe_susie(susie_hc, "susie_eb_res",
                                   z = z_sketch, X = sketch, n = n,
                                   R_finite = B_sketch, R_mismatch = "eb_mix")


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.




$L1
[1] 1074 1155 1180 1268

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: R uncertainty inflation modifies per-variant SER likelihoods, which prevents a consistent model-level ELBO.



plot without title

In [13]:
susie_eb_res$R_finite_diagnostics$R_reliability_flag

[1] FALSE

In [14]:
susie_eb_res$R_finite_diagnostics$prior_variance_flag

NULL

In [8]:
library(tidyverse)
library(patchwork)

lead <- which.max(abs(insample_z))   # top-z variant (the anchor, ~index 1051)

tidy_pip <- function(fit, label, lead) {
  pip <- as.numeric(fit$pip); cs <- fit$sets$cs
  type <- rep(NA_character_, length(pip))
  if (!is.null(cs)) for (nm in names(cs)) {
    members <- cs[[nm]]
    type[members] <- if (lead %in% members) "Lead CS" else "Spurious CS"
  }
  tibble(variant = seq_along(pip), pip = pip, cs_type = type,
         panel = label, n_cs = if (is.null(cs)) 0L else length(cs))
}

pip_df <- bind_rows(
  tidy_pip(insample_res,  "In-sample LD",     lead),
  tidy_pip(susie_rss_res, "Out-of-sample LD", lead),
  tidy_pip(susie_eb_res,  "EB-corrected",     lead)
) %>%
  mutate(panel = factor(panel, levels = c("In-sample LD", "Out-of-sample LD", "EB-corrected")))

lab_map <- pip_df %>% distinct(panel, n_cs) %>%
  mutate(lab = sprintf("%s  (%d CS)", panel, n_cs)) %>% { setNames(.$lab, .$panel) }

cs_cols <- c("Lead CS" = "#08306B", "Spurious CS" = "#E6820E")

fw <- 9; fh <- 8
options(repr.plot.width = fw, repr.plot.height = fh)

p_pip <- ggplot(pip_df, aes(variant, pip)) +
  geom_point(data = filter(pip_df, is.na(cs_type)), colour = "grey78", size = 1) +
  geom_point(data = filter(pip_df, !is.na(cs_type)), aes(colour = cs_type), size = 2.6) +
  facet_grid(panel ~ ., labeller = labeller(panel = lab_map)) +
  scale_colour_manual(values = cs_cols, name = NULL) +
  labs(x = "Variant index", y = "PIP") +
  coord_cartesian(ylim = c(0, 1)) +
  theme_bw(base_size = 16) +
  theme(
    legend.position  = "bottom",
    axis.title       = element_text(size = 18),
    axis.text        = element_text(size = 14),
    strip.text       = element_text(size = 14, face = "bold"),
    strip.background = element_rect(fill = "grey95", colour = NA),
    panel.grid.minor = element_blank()
  )
p_pip

# ggsave("fig5d_pip_tracks.png", p_pip, width = fw, height = fh, dpi = 300)


Attaching package: ‘patchwork’


The following object is masked from ‘package:MASS’:

    area




plot without title

In [24]:
library(tidyverse)

## ============================================================================
## PART 1 — tidy PIPs -> Figure_data
## ============================================================================
data_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/susie_QC/Figure_data"
dir.create(data_dir, showWarnings = FALSE, recursive = TRUE)
data_name <- "PIP_example_locus"

lead <- which.max(abs(insample_z))

tidy_pip <- function(fit, label) {
  pip <- as.numeric(fit$pip)
  cs  <- fit$sets$cs
  cs_id <- rep(NA_character_, length(pip))
  if (!is.null(cs)) for (nm in names(cs)) cs_id[cs[[nm]]] <- nm
  tibble(variant = seq_along(pip), pip = pip, cs = cs_id, panel = label)
}

pip_df <- bind_rows(
  tidy_pip(insample_res,  "In-Sample"),
  tidy_pip(susie_rss_res, "SuSiE-RSS"),
  tidy_pip(susie_eb_res,  "SuSiE-RSS 2.0")
) %>%
  mutate(
    panel = factor(panel, levels = c("In-Sample", "SuSiE-RSS", "SuSiE-RSS 2.0")),
    cs    = factor(cs),
    lead_idx = lead
  )

saveRDS(pip_df,  file.path(data_dir, paste0(data_name, ".rds")))
write_csv(pip_df, file.path(data_dir, paste0(data_name, ".csv")))

## ============================================================================
## PART 2 — plot from stored data
## ============================================================================
fig_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/susie_QC/Figure_output"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

X <-6; Y <- 7
options(repr.plot.width = X, repr.plot.height = Y)

save_fig <- function(plot, name, width = X, height = Y, dir = fig_dir) {
  ggsave(file.path(dir, paste0(name, ".pdf")), plot, width = width, height = height, device = cairo_pdf)
  ggsave(file.path(dir, paste0(name, ".png")), plot, width = width, height = height, dpi = 300)
  invisible(file.path(dir, name))
}

## colorblind-safe credible-set palette (Okabe-Ito, grey reserved for non-CS)
cs_pal <- c("#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00",
            "#56B4E9", "#F0E442", "#6A3D9A", "#B15928")

pip_df <- readRDS(file.path(data_dir, paste0(data_name, ".rds"))) %>% filter(variant < 1300 & variant > 500)
lead   <- unique(pip_df$lead_idx)

p_pip <- ggplot(pip_df, aes(x = variant, y = pip)) +
  # geom_vline(xintercept = lead, linetype = "dashed", colour = "grey55", linewidth = 0.5) +
  geom_point(data = filter(pip_df, is.na(cs)), colour = "grey80", size = 1) +
  geom_point(data = filter(pip_df, !is.na(cs)), aes(fill = cs),
             shape = 21, colour = "grey20", stroke = 0.3, size = 4) +
  facet_grid(panel ~ ., axes = "all_x") +
  scale_fill_manual(values = cs_pal) +
  scale_y_continuous(breaks = c(0, 0.5, 1), expand = expansion(mult = c(0.02, 0.05))) +
  labs(x = "Variant index", y = "PIP", fill = "Credible set", title = "High Light Scatter Reticulocyte% \n chr12:95.6–96.5 Mb") +
  coord_cartesian(ylim = c(0, 1)) +
  theme_bw(base_size = 16) +
  theme(
    panel.grid       = element_blank(),
    panel.border     = element_blank(),
    axis.line        = element_line(colour = "grey20", linewidth = 0.4),
    axis.ticks       = element_line(colour = "grey20", linewidth = 0.4),
    strip.background = element_blank(),
     strip.text.y     = element_text(size = 14, face = "bold", angle = -90), 
    panel.spacing    = unit(0.9, "lines"),
    # legend.position  = "bottom",
          legend.position  = "none",
    legend.title     = element_text(size = 15, face = "bold"),
    legend.text      = element_text(size = 13),
    axis.title       = element_text(size = 18),
    axis.text        = element_text(size = 14),
    text = element_text(family = "sans"),
          plot.title = element_text(size = 15, face = "bold", hjust = 0.5, lineheight = 1.1,
                              margin = margin(b = 6))
  ) +
  guides(fill = guide_legend(override.aes = list(size = 4)))   

## ---- PREVIEW first (check, then uncomment to store) -----------------------
p_pip

# save_fig(p_pip, "PIP_example_locus")

plot without title

In [25]:
save_fig(p_pip, "PIP_example_locus")

In [13]:
in_sample_ld = insample_LD
out_sample_ld = get_cormat(sketch)

In [14]:
index = c(unlist(insample_res$sets$cs), unlist(susie_rss_res$sets$cs))

In [11]:
library(tidyverse)

# Subset LD matrices
ld_in  <- in_sample_ld[index, index]
ld_out <- out_sample_ld[index, index]

# Element-wise difference
ld_diff <- ld_in - ld_out

# Convert to long format
ld_diff_df <- as.data.frame(as.table(ld_diff)) %>%
  rename(
    SNP1 = Var1,
    SNP2 = Var2,
    LD_diff = Freq
  )

# Heatmap
ggplot(ld_diff_df, aes(x = SNP1, y = SNP2, fill = LD_diff)) +
  geom_tile() +
  scale_fill_gradient2(
    low = "blue",
    mid = "white",
    high = "red",
    midpoint = 0,
    name = expression(Delta * "LD")
  ) +
  coord_fixed() +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks = element_blank(),
    panel.grid = element_blank()
  ) +
  labs(
    x = NULL,
    y = NULL,
    title = "LD Difference",
    subtitle = "In-sample LD − Out-of-sample LD"
  )

plot without title

In [18]:
ld_in[1,6]

ld_out[1,6]

[1] -0.1175564

[1] 0.5669654

## Additional locus example

In [30]:
file = list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong/", full.names = T)
res = readRDS(file[15])

In [31]:
file[15]

[1] "/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong//chr14:63989351-66407373.Plateletcrit.rds"

In [52]:
   library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

# ---- helpers ----
safe_susie <- function(.fn, .label, ...) {
  tryCatch(
    withCallingHandlers(
      .fn(...),
      warning = function(w) invokeRestart("muffleWarning")
    ),
    error = function(e) {
      cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
      NULL
    }
  )
}

cs_maxcor <- function(model, Xcorr)
  tryCatch(as.numeric(get_cs_correlation(model, Xcorr = Xcorr, max = TRUE)),
           error = function(e) NA_real_)

# ---- fit loop, one output per (gene, tissue) ----
files = list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong", full.names = T, recursive = T)


f = files[15]

d <- readRDS(f)

insample_z  <- d[["insample_z"]]
insample_LD <- d[["insample_LD"]]
z_sketch    <- d[["z_sketch"]]
sketch      <- d[["sketch"]]
sketch_LD   <- d[["sketch_LD"]]
n           <- d[["n"]]
B_insample  <- d[["B_insample"]]
B_sketch    <- d[["B_sketch"]]

# ---- in-sample panel ----
insample_res         <- safe_susie(susie_hc, "insample_res",
                                   z = insample_z, R = insample_LD, n = n)
susie_plot(insample_res, "PIP")

insample_res$sets$cs

susie_rss_res        <- safe_susie(susie_hc, "susie_rss_res",
                                       z = z_sketch, X = sketch, n = n)

susie_eb_res         <- safe_susie(susie_hc, "susie_eb_res",
                                   z = z_sketch, X = sketch, n = n,
                                   R_finite = B_sketch, R_mismatch = "eb_mix")


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.




$L1
[1] 6281 6282 6289 6309

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: R uncertainty inflation modifies per-variant SER likelihoods, which prevents a consistent model-level ELBO.



plot without title

In [53]:
insample_res$sets$cs

$L1
[1] 6281 6282 6289 6309

In [54]:
susie_rss_res$sets$cs

$L1
[1] 6281 6282 6289 6309

$L2
 [1] 2958 2964 3010 3030 3031 3040 3048 3100 3130 3155 3204 3239 3305 3319 3324
[16] 3352 3363 3391 3394 3396 3398 3402 3407 3409 3472 3473 3474 3499 3530 3545

In [55]:
susie_eb_res$sets$cs

$L1
[1] 6281 6282 6289 6309

In [36]:
susie_eb_res$R_finite_diagnostics$R_reliability_flag

[1] FALSE

In [37]:
susie_eb_res$R_finite_diagnostics$prior_variance_flag

NULL

In [56]:
index = c(unlist(insample_res$sets$cs), unlist(susie_rss_res$sets$cs))

In [57]:
library(tidyverse)

# Subset LD matrices
ld_in  <- in_sample_ld[index, index]
ld_out <- out_sample_ld[index, index]

# Element-wise difference
ld_diff <- ld_in - ld_out

# Convert to long format
ld_diff_df <- as.data.frame(as.table(ld_diff)) %>%
  rename(
    SNP1 = Var1,
    SNP2 = Var2,
    LD_diff = Freq
  )

# Heatmap
ggplot(ld_diff_df, aes(x = SNP1, y = SNP2, fill = LD_diff)) +
  geom_tile() +
  scale_fill_gradient2(
    low = "blue",
    mid = "white",
    high = "red",
    midpoint = 0,
    name = expression(Delta * "LD")
  ) +
  coord_fixed() +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks = element_blank(),
    panel.grid = element_blank()
  ) +
  labs(
    x = NULL,
    y = NULL,
    title = "LD Difference",
    subtitle = "In-sample LD − Out-of-sample LD"
  )

ERROR: Error in in_sample_ld[index, index]: subscript out of bounds
